# MotorPH Terminal Assessment - Final Analysis Notebook

**Learner:** Kera Froilan Montejo  
**Section:** H2101  
**Course:** MO-IT106 Data Analytics Fundamentals

This final notebook integrates Milestone 1, Milestone 2, inferential analysis, and the instructor's final feedback.

### Final feedback updates addressed
1. Added external economic context for the June-August sales decline.
2. Added a sensitivity check for the 19 original unit-price mismatches.
3. Added specific proposed Scooter margin decision benchmarks.
4. Added two-tailed Mann-Whitney robustness tests alongside the original one-tailed tests.

> Important: the 19 price-mismatched records were **not excluded** from the finalized MS2 dataset. Their unit prices were corrected using the Product master. The exclusion analysis below is only a robustness test.


In [1]:
from MotorPH_Terminal_Assessment_Analysis_FINAL import (
    load_data, descriptive_analysis, price_mismatch_sensitivity,
    inferential_analysis, proposed_scooter_margin_benchmarks,
    create_supporting_charts
)

products, sales, products_path, sales_path = load_data()

print("Products file:", products_path.name)
print("Sales file:", sales_path.name)
print("Valid Q3 sales rows:", len(sales))
print("Duplicate rows:", sales.duplicated().sum())


Products file: MotorPH_Products_Preprocessed.csv
Sales file: MotorPH_Sales_MS2_Cleaned.csv
Valid Q3 sales rows: 987
Duplicate rows: 0


## 1. Descriptive Analysis

The descriptive stage establishes the product portfolio and the Q3 sales pattern before inferential testing. The report uses **listed product value** rather than claiming the sum of unit prices is actual physical inventory value because the Product dataset has no stock-on-hand quantity field.


In [2]:
product_counts, monthly_summary = descriptive_analysis(products, sales)
monthly_summary



=== DESCRIPTIVE ANALYSIS ===
Products: 50
Categories: 17
Average unit price: PHP 257,872.00
Median unit price: PHP 204,950.00
Minimum unit price: PHP 48,000.00
Maximum unit price: PHP 995,000.00
Sum of listed unit prices: PHP 12,893,600.00

Monthly sales:
         Units_Sold     Revenue  Transactions
Month                                        
2025-06        6080  1519450700           341
2025-07        5833  1488398700           330
2025-08        5610  1432349900           316

Total Q3 units sold: 17,523
Total Q3 revenue: PHP 4,440,199,300.00


,Units_Sold,Revenue,Transactions
Month,,,
2025-06,6080,1519450700,341
2025-07,5833,1488398700,330
2025-08,5610,1432349900,316


## 2. Sensitivity Analysis - 19 Price-Mismatched Rows

The finalized MS2 workflow corrected the 19 inconsistent unit prices using the Product master and then recalculated revenue. To test whether those records could distort the trend, I temporarily exclude them and compare the monthly revenue pattern with the official corrected analysis.

A robust conclusion should remain directionally similar even when those rows are removed.


In [3]:
sensitivity = price_mismatch_sensitivity(products)
sensitivity



=== PRICE-MISMATCH SENSITIVITY CHECK ===
The 19 rows were corrected in the final dataset, not excluded. This robustness test temporarily excludes them.
         Revenue_All_Corrected_Rows  ...  MoM_Excluding_Mismatches_Pct
Month                                ...                              
2025-06                  1519450700  ...                           NaN
2025-07                  1488398700  ...                        -2.553
2025-08                  1432349900  ...                        -3.708

[3 rows x 9 columns]

June-to-August revenue change with corrected rows: -5.73%
June-to-August revenue change excluding mismatch rows: -6.17%
Conclusion: the June > July > August decline remains in both cases, so the trend is not driven by the 19 price-mismatch rows.


,Revenue_All_Corrected_Rows,Revenue_Excluding_19_Mismatches,Mismatch_Rows,Transactions,Revenue_Difference,Revenue_Difference_Pct,Mismatch_Rate_Pct,MoM_All_Corrected_Pct,MoM_Excluding_Mismatches_Pct
Month,,,,,,,,,
2025-06,1519450700,1495845000,7,341,23605700,1.553568,2.052786,NaN,NaN
2025-07,1488398700,1457656700,5,330,30742000,2.065441,1.515152,-2.043633,-2.552958
2025-08,1432349900,1403609800,7,316,28740100,2.006500,2.215190,-3.765711,-3.707793


### Sensitivity interpretation

The mismatch records are distributed across all three months rather than concentrated in only one month. Excluding them reduces each month's revenue by only about 1.6%-2.1%. More importantly, the pattern remains **June > July > August**.

- Corrected-data June-to-August revenue change: about **-5.73%**
- Excluding mismatch rows: about **-6.17%**

Therefore, the downward revenue trend is not being created by the 19 price-mismatched transactions.


## 3. Inferential Analysis and Two-Tailed Robustness Checks

The original assessment questions are directional, so the primary tests remain **one-tailed**. The instructor asked how a two-tailed test would affect the finding. A two-tailed Mann-Whitney test asks whether the distributions differ **in either direction**, rather than testing only whether the later month is greater.

The two-tailed results do not change the final decision, but they provide a more symmetric robustness check.


In [4]:
inferential = inferential_analysis(products, sales)
inferential["results_table"]



=== INFERENTIAL ANALYSIS (alpha = 0.05) ===
Q1 July Shapiro-Wilk: W=0.963, p=0.116
Q2 one-tailed: U=352.0, p=0.949
Q2 two-tailed robustness: U=352.0, p=0.105, rank-biserial=-0.243
Q3 August Shapiro-Wilk: W=0.965, p=0.143
Q4 one-tailed: U=792.0, p=0.898
Q4 two-tailed robustness: U=792.0, p=0.206, rank-biserial=-0.162


,Question,Test,Statistic,p_value,Decision
0,Q1 July product sales normality,Shapiro-Wilk,0.962755,0.115961,Fail to reject H0
1,Q2 July retail sales > June,"Mann-Whitney U (one-tailed, greater)",352.000000,0.949224,Fail to reject H0
2,Q2 robustness: July retail differs from June,Mann-Whitney U (two-tailed),352.000000,0.104600,Fail to reject H0
3,Q3 August product sales normality,Shapiro-Wilk,0.964942,0.143125,Fail to reject H0
4,Q4 August units > previous months,"Mann-Whitney U (one-tailed, greater)",792.000000,0.898343,Fail to reject H0
5,Q4 robustness: August differs from previous mo...,Mann-Whitney U (two-tailed),792.000000,0.206264,Fail to reject H0


### Two-tailed interpretation

- **July retail vs June:** one-tailed p = **0.949**; two-tailed p = **0.105**.
- **August vs June-July daily units:** one-tailed p = **0.898**; two-tailed p = **0.206**.

Both two-tailed p-values remain above 0.05, so there is still insufficient evidence of a statistically significant difference at the 5% level. The two-tailed values are smaller because the observed data moved in the opposite direction from the original "greater than" alternative.

This does **not** mean the months are identical. It means the available daily observations do not provide enough evidence at alpha = 0.05 to conclude a statistically significant difference in either direction.


## 4. Proposed Scooter Margin Benchmarks

The current datasets contain selling prices but **not unit cost/COGS**, so actual gross margin cannot be calculated. The following thresholds are proposed management rules to use once cost data become available.

A major Scooter-lineup change should not be based only on its +0.90 percentage-point lineup-to-sales gap because the category is relatively aligned with demand and leads both units and revenue.


In [5]:
margin_benchmarks = proposed_scooter_margin_benchmarks()
margin_benchmarks


,Decision Zone,Gross Margin Benchmark,Additional Condition,Action
0,Maintain,"Scooter GM% at/above portfolio GM%, or no more...",Gross profit per unit at/above portfolio median,Maintain lineup; continue monitoring sell-through
1,Review,Scooter GM% 3-5 pp below portfolio GM%,Persists for 2 consecutive months,"Review pricing, promotions, product mix, and a..."
2,Significant change,Scooter GM% >5 pp below portfolio GM%,Persists for 2 months AND sell-through <80% of...,Consider meaningful assortment/replenishment r...


### Margin decision rule

I would maintain the Scooter lineup when category gross margin percentage is at or above the overall portfolio gross margin, or no more than **2 percentage points below it**, while gross profit per unit remains at or above the portfolio median.

I would trigger a management review if Scooter gross margin remains **3-5 percentage points below** the portfolio gross margin for two consecutive months.

I would consider a significant assortment or replenishment reduction only if Scooter gross margin is **more than 5 percentage points below** the portfolio gross margin for two consecutive months **and** sell-through falls below **80% of the portfolio median or management target**. These are proposed operating thresholds, not margins observed in the current dataset.


## 5. External Economic Context

The MotorPH dataset alone cannot establish *why* unit sales fell from June to August, so external indicators should be treated as context rather than causal proof.

Official Philippine sources provide several plausible factors:

- **Inflation:** PSA reported headline inflation of **1.4% in June 2025**, **0.9% in July**, and **1.5% in August**. Because inflation first eased and then rose while MotorPH units declined each month, inflation alone does not explain the continuous sales decrease.
- **Business sentiment and seasonality:** BSP's Q3 2025 Business Expectations Survey reported that current-quarter business confidence fell from **28.8% in Q2 to 23.2% in Q3**. Respondents cited seasonal slack in demand, the rainy and typhoon season, lower customer foot traffic, and a shift in spending toward school and household needs.
- **Broader economy:** PSA reported Q2 2025 GDP growth of **5.5%**, while wholesale and retail trade and repair of motor vehicles and motorcycles grew **5.1%** year-on-year. This means MotorPH's Q3 decline occurred even though the broader economy was still expanding.
- **Interest rates:** BSP reduced the target reverse repurchase rate to **5.0% on 28 August 2025**. Any effect on vehicle financing would likely occur with a lag and therefore cannot be assumed to explain the June-August pattern.

### Sources
- Philippine Statistics Authority, June 2025 CPI: https://psa.gov.ph/content/summary-inflation-report-consumer-price-index-2018100-june-2025
- Philippine Statistics Authority, July 2025 CPI: https://psa.gov.ph/content/summary-inflation-report-consumer-price-index-2018100-july-2025
- Philippine Statistics Authority, August 2025 CPI: https://psa.gov.ph/content/summary-inflation-report-consumer-price-index-2018100-august-2025
- BSP, Q3 2025 Business Expectations Survey: https://www.bsp.gov.ph/Lists/Business%20Expectations%20Report/Attachments/23/BES_3qtr2025.pdf
- PSA, Q2 2025 GDP: https://psa.gov.ph/content/gdp-expands-55-percent-second-quarter-2025
- BSP, August 2025 Monetary Policy Report: https://www.bsp.gov.ph/SitePages/PriceStability/FullReportMPR/MonetaryPolicyReport_Full_August2025.aspx

### Interpretation
These external factors make the report more complete, but they should not be presented as proven causes. A causal analysis would require MotorPH-specific variables such as promotions, financing approvals, customer traffic, regional weather, competitor prices, stock availability, and customer-income or financing data.


## 6. Final Business Interpretation

The updated evidence supports the same core recommendations while strengthening their reliability:

- The June-to-August sales decline remains even after a price-mismatch sensitivity test.
- Naked Bike and Underbone remain the clearest assortment-risk signals.
- Scooter should be maintained unless future **margin + sell-through** evidence justifies a major change.
- Two-tailed statistical checks do not overturn the inferential conclusions.
- External economic and seasonal conditions may help explain the timing of the decline, but MotorPH-specific causal data are still needed before assigning a cause.


In [6]:
create_supporting_charts(products, sales, inferential)
print("Final notebook analysis complete.")


Final notebook analysis complete.
